# Projet : VNF Chain Optimization
Ce notebook formalise la logique et la modélisation mathématique pour résoudre le projet d'optimisation (placements de VNF sur des serveurs).

## Phase 1 : Formulation du MILP (Solution Exacte)
L'objectif est de trouver un placement de $n$ fonctions réseau (VNF) sur $m$ serveurs physiques qui minimise la charge maximale parmi tous les serveurs (le goulot d'étranglement ou cycle time $CT$).

### 1. Variables de Décision
- $x_{jk} \in \{0, 1\}$ : Vaut 1 si la VNF $j$ est placée sur le serveur $k$, 0 sinon.
- $s_j \in \{1, \dots, m\}$ : Indice du serveur sur lequel la VNF $j$ est placée.
- $CT \ge 0$ : Le "Cycle Time" ou latence du goulot d'étranglement (charge maximale d'un serveur).

### 2. Fonction Objectif
Minimiser $CT$

### 3. Contraintes
1. **Unicité du placement** : Chaque VNF $j$ doit être placée sur exactement un serveur.
   $$\sum_{k=1}^m x_{jk} = 1 \quad \forall j \in J$$
2. **Définition de l'indice du serveur** : Lien entre les variables binaires $x$ et l'indice $s_j$.
   $$s_j = \sum_{k=1}^m k \cdot x_{jk} \quad \forall j \in J$$
3. **Respect de l'ordre (Graphe des précédences)** : Pour chaque arc $(i, j) \in P$, la VNF $i$ doit être placée sur le même serveur ou un serveur précédent par rapport à $j$.
   $$s_i \le s_j \quad \forall (i, j) \in P$$
4. **Définition de la latence maximale du serveur ($CT$)** : La somme des temps de traitement appliqués sur n'importe quel serveur $k$ ne doit pas dépasser le bottleneck.
   $$\sum_{j=1}^n t_j \cdot x_{jk} \le CT \quad \forall k \in K$$

In [3]:
# Squelette de code pour Gurobi (Phase 1)
import gurobipy as gp
from gurobipy import GRB

def solve_phase1(J, K, P, t):
    """
    J : liste d'entiers (fonctions VNF)
    K : liste d'entiers (serveurs)
    P : liste de tuples (i, j) représentant les précédences
    t : dictionnaire des temps de traitement (t_j)
    """
    model = gp.Model("VNF_Placement")

    # --- Variables ---
    x = model.addVars(J, K, vtype=GRB.BINARY, name="x")
    s = model.addVars(J, vtype=GRB.INTEGER, lb=1, ub=len(K), name="s")
    CT = model.addVar(vtype=GRB.CONTINUOUS, lb=0, name="CT")

    # --- Contraintes ---
    # 1. Unicité du placement
    model.addConstrs((x.sum(j, '*') == 1 for j in J), name="unicite")

    # 2. Définition de s_j
    model.addConstrs((s[j] == gp.quicksum(k * x[j, k] for k in K) for j in J), name="def_s")

    # 3. Précédences (DAG)
    model.addConstrs((s[i] <= s[j] for i, j in P), name="precedences")

    # 4. Goulot d'étranglement (Charge max)
    model.addConstrs((gp.quicksum(t[j] * x[j, k] for j in J) <= CT for k in K), name="bottleneck")

    # --- Objectif ---
    model.setObjective(CT, GRB.MINIMIZE)

    return model

# Exemple d'utilisation avec données de test
if __name__ == "__main__":
    # Données d'exemple
    J = [0, 1, 2, 3]  # 4 VNF
    K = [0, 1, 2]     # 3 serveurs (indices 0, 1, 2)
    P = [(0, 1), (1, 2), (0, 3)]  # Précédences: 0->1, 1->2, 0->3
    t = {0: 5, 1: 3, 2: 4, 3: 2}  # Temps de traitement

    # Résoudre le modèle
    model = solve_phase1(J, K, P, t)
    model.optimize()

    # Afficher les résultats
    if model.status == GRB.OPTIMAL:
        print(f"Cycle Time optimal: {model.objVal}")
        print("\nPlacement des VNF:")
        for j in J:
            for k in K:
                if model.getVarByName(f"x[{j},{k}]").X > 0.5:
                    print(f"VNF {j} -> Serveur {k}")
        print("\nCharge des serveurs:")
        for k in K:
            load = sum(t[j] * model.getVarByName(f"x[{j},{k}]").X for j in J)
            print(f"Serveur {k}: {load}")
    else:
        print("Aucune solution optimale trouvée")

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.3.0 25D2128)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 14 rows, 17 columns and 45 nonzeros (Min)
Model fingerprint: 0x27b38265
Model has 1 linear objective coefficients
Variable types: 1 continuous, 16 integer (12 binary)
Coefficient statistics:
  Matrix range     [1e+00, 5e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 3e+00]
  RHS range        [1e+00, 1e+00]

Presolve removed 14 rows and 17 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of 8 available processors)

Solution count 1: 7 

Optimal solution found (tolerance 1.00e-04)
Best objective 7.000000000000e+00, best bound 7.000000000000e+00, gap 0.0000%
Cycle Time optimal: 7.0

Placement des VNF:
VNF 0 -> Serveur 1
VNF 1 -> Serveur 2
VNF 2 -> Serveur 

## Phase 2 : Heuristiques et Métaheuristiques
Les modèles exacts (Phase 1) ne passent pas à l'échelle sur 50 ou 100 variables binaires avec des graphes denses. Nous implémenterons 3 algorithmes approchés pour trouver de bonnes solutions très rapidement :

### 1. Heuristique Gloutonne (List Scheduling)
**Logique** : 
- On trie topologiquement le graphe des VNF pour garantir que les précédences seront respectées si on assigne séquentiellement.
- On se fixe une limite de capacité cible par serveur (par exemple : $\frac{\sum t_j}{m}$).
- On remplit le serveur courant VNF par VNF. Dès qu'ajouter la VNF dépasse la limite, on passe au serveur suivant.
**Avantage** : Extrêmement rapide ($O(n \log n)$).
**Problème** : Donne une solution potentiellement sous-optimale si le tri topologique n'est pas idéal ou la limite mal choisie.

### 2. Recuit Simulé (Simulated Annealing) - Métaheuristique
**Logique** : 
- On génère une solution initiale valide (ex: avec l'heuristique Gloutonne).
- **Voisinage** : À chaque itération, on sélectionne une VNF au hasard et on tente de la déplacer sur un serveur adjacent ($s_j + 1$ ou $s_j - 1$). On vérifie strictement que les précédences $s_i \le s_j$ ne sont pas violées.
- **Fonction de coût (Fitness)** : L'énergie $E = CT$ (la capacité max du système dans l'état actuel).
- Règle d'acceptation : Si le voisin baisse le cycle time ($E' \le E$), on le garde. S'il l'augmente, on l'accepte tout de même avec une probabilité $P = \exp(\frac{E - E'}{T})$, où $T$ est la température du système qui décroît progressivement.
**Avantage** : Peut éviter les minima locaux en acceptant occasionnellement de "mauvais" placements au début du processus.

### 3. Algorithme Génétique (Genetic Algorithm) - Métaheuristique
**Logique** : 
- **Population** : Chaque individu (chromosome) est un vecteur d'entiers de taille $n$, où la $j$-ème case désigne le serveur $s_j$ de la VNF $j$.
- **Sélection** : On privilégie la reproduction des chromosomes ayant la plus faible latence maximale $CT$.
- **Croisement (Crossover) et Mutation** : On mixe le placement de deux parents pour créer des enfants. Puis, on mute en déplaçant aléatoirement quelques VNFs avec une faible probabilité.
- **Gestion des Contraintes (Pénalités)** : Les croisements peuvent produire des individus qui violent le DAG de précédences. De tels individus reçoivent une "Fitness" artificiellement désastreuse (fortement pénalisée) pour qu'ils soient naturellement éliminés au tour suivant.
**Avantage** : Explore massivement de multiples configurations asynchrones très différentes, capable de trouver des structures que le recuit n'attraperait pas.

## Résultats de la Phase 1

La Phase 1 est maintenant complète et fonctionnelle. Le modèle MILP trouve la solution optimale pour le placement des VNF :

- **Cycle Time optimal** : 7.0 unités de temps
- **Placement** : VNF 0 et 3 sur le Serveur 1, VNF 1 et 2 sur le Serveur 2
- **Charges** : Serveur 0 inutilisé (0.0), Serveurs 1 et 2 à pleine capacité (7.0 chacun)

Le modèle respecte toutes les contraintes :
- Chaque VNF est placée sur exactement un serveur
- Les précédences sont respectées (0→1→2, 0→3)
- Le goulot d'étranglement (charge maximale) est minimisé

Cette approche exacte garantit la solution optimale mais ne scale pas bien pour de gros problèmes (50+ VNF). La Phase 2 implémentera des heuristiques pour des instances plus grandes.

## Phase 3 : Implémentation des heuristiques et métaheuristiques

La Phase 3 se concentre sur l'application effective des heuristiques abordées en Phase 2 :
- Heuristique gloutonne (List Scheduling)
- Recuit simulé (Simulated Annealing)
- Algorithme génétique (Genetic Algorithm)

Le but est d'obtenir des solutions de bonne qualité rapidement sur des instances plus larges, même si elles ne sont pas optimales à 100 %.


In [2]:
import random
import math

# --- Utilities ---

def evaluate_solution(J, K, t, P, assignment):
    """Retourne CT et charge par serveur; assignment est un dict j->k."""
    # vérifier précédences
    for i, j in P:
        if assignment[i] > assignment[j]:
            return float('inf'), None
    loads = {k: 0.0 for k in K}
    for j in J:
        loads[assignment[j]] += t[j]
    CT = max(loads.values())
    return CT, loads

# --- Heuristique gloutonne (list scheduling) ---

def greedy_assignment(J, K, t, P):
    # ordre topologique simplifié basé sur nombre de prédécesseurs
    pred_count = {j: 0 for j in J}
    succ = {j: [] for j in J}
    for i, j in P:
        pred_count[j] += 1
        succ[i].append(j)

    queue = [j for j in J if pred_count[j] == 0]
    order = []
    while queue:
        u = min(queue, key=lambda x: t[x])
        queue.remove(u)
        order.append(u)
        for v in succ[u]:
            pred_count[v] -= 1
            if pred_count[v] == 0:
                queue.append(v)

    assignment = {j: None for j in J}
    loads = {k: 0.0 for k in K}

    for j in order:
        best_k = min(K, key=lambda k: loads[k])
        assignment[j] = best_k
        loads[best_k] += t[j]

    CT, _ = evaluate_solution(J, K, t, P, assignment)
    return assignment, CT, loads

# --- Recuit simulé (simulated annealing) ---

def simulated_annealing(J, K, t, P, init_assignment, init_CT, T0=100.0, alpha=0.95, max_iter=500):
    current = init_assignment.copy()
    curr_CT = init_CT
    best = current.copy()
    best_CT = curr_CT

    T = T0
    for it in range(max_iter):
        j = random.choice(J)
        k_new = random.choice([k for k in K if k != current[j]])
        cand = current.copy()
        cand[j] = k_new
        cand_CT, _ = evaluate_solution(J, K, t, P, cand)

        if cand_CT < curr_CT or random.random() < math.exp((curr_CT - cand_CT) / max(T, 1e-9)):
            current = cand
            curr_CT = cand_CT
            if curr_CT < best_CT:
                best = current.copy()
                best_CT = curr_CT

        T *= alpha

    return best, best_CT

# --- Algorithme génétique (GA) simplifié ---

def genetic_algorithm(J, K, t, P, population_size=50, generations=100, mutation_rate=0.15):
    def random_solution():
        return {j: random.choice(K) for j in J}

    def fitness(sol):
        ct, _ = evaluate_solution(J, K, t, P, sol)
        return 1.0 / (1.0 + ct)

    pop = [random_solution() for _ in range(population_size)]

    for gen in range(generations):
        scored = [(fitness(sol), sol) for sol in pop]
        scored.sort(reverse=True, key=lambda x: x[0])
        pop = [sol for _, sol in scored[: population_size // 2]]

        while len(pop) < population_size:
            p1 = random.choice(pop)
            p2 = random.choice(pop)
            child = {j: p1[j] if random.random() < 0.5 else p2[j] for j in J}

            if random.random() < mutation_rate:
                j = random.choice(J)
                child[j] = random.choice(K)

            pop.append(child)

    best = min(pop, key=lambda sol: evaluate_solution(J, K, t, P, sol)[0])
    best_CT, best_loads = evaluate_solution(J, K, t, P, best)
    return best, best_CT, best_loads

# --- Exemple de Phase 3 ---
if __name__ == "__main__":
    J = [0, 1, 2, 3, 4, 5]
    K = [0, 1, 2, 3]
    P = [(0, 1), (0, 2), (1, 3), (2, 3), (3, 4)]
    t = {0: 10, 1: 8, 2: 7, 3: 6, 4: 5, 5: 4}

    print("--- Heuristique gloutonne ---")
    assign_g, CT_g, load_g = greedy_assignment(J, K, t, P)
    print("CT", CT_g, "assign", assign_g, "load", load_g)

    print("--- Recuit simulé ---")
    assign_sa, CT_sa = simulated_annealing(J, K, t, P, assign_g, CT_g, T0=50, alpha=0.90, max_iter=1000)
    _, load_sa = evaluate_solution(J, K, t, P, assign_sa)
    print("CT", CT_sa, "assign", assign_sa, "load", load_sa)

    print("--- Algorithme génétique ---")
    assign_ga, CT_ga, load_ga = genetic_algorithm(J, K, t, P)
    print("CT", CT_ga, "assign", assign_ga, "load", load_ga)


--- Heuristique gloutonne ---
CT inf assign {0: 1, 1: 3, 2: 2, 3: 0, 4: 2, 5: 0} load {0: 10.0, 1: 10.0, 2: 12.0, 3: 8.0}
--- Recuit simulé ---
CT inf assign {0: 1, 1: 3, 2: 2, 3: 0, 4: 2, 5: 0} load None
--- Algorithme génétique ---
CT 11.0 assign {0: 0, 1: 2, 2: 1, 3: 3, 4: 3, 5: 1} load {0: 10.0, 1: 11.0, 2: 8.0, 3: 11.0}


In [5]:
## 4. Intégration avec les données réelles (.alb)

# Parseur pour fichiers .alb
def parse_alb_file(filepath):
    """Parse un fichier .alb et retourne J, K, P, t, cycle_time_limit"""
    data = {}
    current_section = None
    
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('<end>'):
                continue
            
            # Détecter sections
            if line.startswith('<number of tasks>'):
                current_section = 'tasks_count'
                continue
            elif line.startswith('<cycle time>'):
                current_section = 'cycle_time'
                continue
            elif line.startswith('<order strength>'):
                current_section = 'order_strength'
                continue
            elif line.startswith('<task times>'):
                current_section = 'task_times'
                data['task_times'] = {}
                continue
            elif line.startswith('<precedence relations>'):
                current_section = 'precedences'
                data['precedences'] = []
                continue
            
            # Parser le contenu
            if current_section == 'tasks_count':
                data['n_tasks'] = int(line)
                current_section = None
            elif current_section == 'cycle_time':
                data['cycle_time'] = int(line)
                current_section = None
            elif current_section == 'order_strength':
                # Gérer virgule décimale (format français)
                line_clean = line.replace(',', '.')
                data['order_strength'] = float(line_clean)
                current_section = None
            elif current_section == 'task_times':
                parts = line.split()
                if len(parts) == 2:
                    task_id = int(parts[0]) - 1  # 0-indexed
                    task_time = int(parts[1])
                    data['task_times'][task_id] = task_time
            elif current_section == 'precedences':
                parts = line.split(',')
                if len(parts) == 2:
                    i = int(parts[0]) - 1  # 0-indexed
                    j = int(parts[1]) - 1
                    data['precedences'].append((i, j))
    
    # Construire structures
    J = list(range(data['n_tasks']))
    K = list(range(4))  # On assume 4 serveurs de base (adaptable)
    P = data['precedences']
    t = data['task_times']
    
    return J, K, P, t, data['cycle_time'], data['order_strength']

# Tester le parseur
filepath = "/Users/olivierrecher/Documents/IMT/S4/Opti/Instances/instance_n=20_1.alb"
J, K, P, t, ct_limit, os = parse_alb_file(filepath)

print(f"✓ Instance chargée:")
print(f"  - Nombre de VNF (J): {len(J)}")
print(f"  - VNF: {J}")
print(f"  - Nombre de serveurs (K): {len(K)}")
print(f"  - Serveurs: {K}")
print(f"  - Nombre de précédences: {len(P)}")
print(f"  - Précédences (premiers): {P[:5]}")
print(f"  - Temps min/max: {min(t.values())}/{max(t.values())}")
print(f"  - Cycle time limite: {ct_limit}")
print(f"  - Order strength: {os}")


✓ Instance chargée:
  - Nombre de VNF (J): 20
  - VNF: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
  - Nombre de serveurs (K): 4
  - Serveurs: [0, 1, 2, 3]
  - Nombre de précédences: 16
  - Précédences (premiers): [(0, 5), (1, 6), (3, 7), (4, 8), (5, 9)]
  - Temps min/max: 34/282
  - Cycle time limite: 1000
  - Order strength: 0.268


In [6]:
## Résolution Phase 1 sur instance réelle

# Adapter et résoudre le modèle Phase 1
def solve_phase1_real_instance(J, K, P, t):
    """Résoudre le modèle MILP exact sur instance réelle"""
    model = gp.Model("VNF_Placement_Real")
    
    # --- Variables ---
    x = model.addVars(J, K, vtype=GRB.BINARY, name="x")
    s = model.addVars(J, vtype=GRB.INTEGER, lb=0, ub=len(K)-1, name="s")
    CT = model.addVar(vtype=GRB.CONTINUOUS, lb=0, name="CT")
    
    # --- Contraintes ---
    # 1. Unicité du placement
    model.addConstrs((x.sum(j, '*') == 1 for j in J), name="unicite")
    
    # 2. Définition de s_j (indice du serveur)
    model.addConstrs((s[j] == gp.quicksum(k * x[j, k] for k in K) for j in J), name="def_s")
    
    # 3. Précédences
    model.addConstrs((s[i] <= s[j] for i, j in P), name="precedences")
    
    # 4. Goulot d'étranglement
    model.addConstrs((gp.quicksum(t[j] * x[j, k] for j in J) <= CT for k in K), name="bottleneck")
    
    # --- Objectif ---
    model.setObjective(CT, GRB.MINIMIZE)
    
    # Optimiser
    model.optimize()
    
    return model, x, s, CT

# Résoudre
print("🔧 Résolution en cours...")
model, x, s, CT = solve_phase1_real_instance(J, K, P, t)

# Afficher résultats
print("\n" + "="*60)
if model.status == GRB.OPTIMAL:
    print(f"✓ SOLUTION OPTIMALE TROUVÉE")
    print(f"  CT optimal: {model.objVal:.1f}")
    print(f"\n📊 Placement des VNF:")
    for j in J:
        for k in K:
            if x[j, k].X > 0.5:
                print(f"   VNF {j:2d} (temps={t[j]:3d}) → Serveur {k}")
    
    print(f"\n📈 Charge par serveur:")
    for k in K:
        load = sum(t[j] * x[j, k].X for j in J)
        utilisation = (load / model.objVal * 100) if model.objVal > 0 else 0
        print(f"   Serveur {k}: {load:6.1f} [{utilisation:5.1f}%]")
    
    print(f"\n⏱️ Temps de résolution: {model.Runtime:.3f}s")
    print(f"  Itérations simplex: {model.IterCount}")
    print("="*60)
else:
    print(f"✗ Aucune solution trouvée (status: {model.status})")
    print("="*60)


🔧 Résolution en cours...
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.3.0 25D2128)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 60 rows, 101 columns and 276 nonzeros (Min)
Model fingerprint: 0xdf61a7d3
Model has 1 linear objective coefficients
Variable types: 1 continuous, 100 integer (80 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+02]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 3e+00]
  RHS range        [1e+00, 1e+00]

Found heuristic solution: objective 2882.0000000
Presolve removed 12 rows and 12 columns
Presolve time: 0.00s
Presolved: 48 rows, 89 columns, 264 nonzeros
Variable types: 0 continuous, 89 integer (80 binary)

Root relaxation: objective 7.205000e+02, 60 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/N

In [8]:
## Comparaison : Phase 1 (Optimal) vs Phase 3 (Heuristiques)

print("\n" + "🔍 COMPARAISON EXACT vs HEURISTIQUES".center(70, "="))

# Phase 1 - Solution optimale
ct_optimal = model.objVal
print(f"\n✓ Phase 1 (OPTIMAL):        CT = {ct_optimal:.1f}")

# Phase 3 - Heuristique gloutonne
assign_greedy, ct_greedy, load_greedy = greedy_assignment(J, K, t, P)
print(f"• Phase 3 (Greedy):         CT = {ct_greedy:.1f}  |  Écart: {(ct_greedy-ct_optimal)/ct_optimal*100:+.1f}%")

# Phase 3 - Recuit simulé
assign_sa, ct_sa = simulated_annealing(J, K, t, P, assign_greedy, ct_greedy, T0=100, alpha=0.95, max_iter=500)
_, load_sa = evaluate_solution(J, K, t, P, assign_sa)
print(f"• Phase 3 (Simulated Ann.):  CT = {ct_sa:.1f}  |  Écart: {(ct_sa-ct_optimal)/ct_optimal*100:+.1f}%")

# Phase 3 - Algorithme génétique
assign_ga, ct_ga, load_ga = genetic_algorithm(J, K, t, P, population_size=50, generations=100)
print(f"• Phase 3 (Genetic Algo):    CT = {ct_ga:.1f}  |  Écart: {(ct_ga-ct_optimal)/ct_optimal*100:+.1f}%")

print("\n" + "="*70)
print(f"Meilleure heuristique: ", end="")
results = [("Greedy", ct_greedy), ("SA", ct_sa), ("GA", ct_ga)]
best_name, best_ct = min(results, key=lambda x: x[1])
ratio = (best_ct - ct_optimal) / ct_optimal * 100
print(f"{best_name} (CT={best_ct:.1f}, écart={ratio:+.1f}%)")
print("="*70)



=================🔍 COMPARAISON EXACT vs HEURISTIQUES==================

✓ Phase 1 (OPTIMAL):        CT = 722.0
• Phase 3 (Greedy):         CT = inf  |  Écart: +inf%
• Phase 3 (Simulated Ann.):  CT = inf  |  Écart: +inf%
• Phase 3 (Genetic Algo):    CT = inf  |  Écart: +inf%

Meilleure heuristique: Greedy (CT=inf, écart=+inf%)


In [9]:
## 📊 Analyse détaillée de la solution optimale Phase 1

print("\n\n" + "ANALYSE DÉTAILLÉE - SOLUTION OPTIMALE".center(70, "="))

# Afficher le plaçage
print(f"\n▶ Placement optimale des 20 VNF sur {len(K)} serveurs:")
placement_by_server = {}
for j in J:
    for k in K:
        if x[j, k].X > 0.5:
            if k not in placement_by_server:
                placement_by_server[k] = []
            placement_by_server[k].append(j)

for k in K:
    if k in placement_by_server:
        vnfs = placement_by_server[k]
        load = sum(t[j] for j in vnfs)
        print(f"  Serveur {k}: VNF {vnfs}  (Load: {load})")
    else:
        print(f"  Serveur {k}: [vide]")

# Analyse des précédences
print(f"\n▶ Vérification des précédences (validité):")
violations = []
for i, j in P:
    if s[i].X > s[j].X:
        violations.append(f"  ✗ {i}→{j} violée (s[{i}]={s[i].X} > s[{j}]={s[j].X})")
    else:
        print(f"  ✓ {i}→{j} respectée (s[{i}]={s[i].X} ≤ s[{j}]={s[j].X})", end="")

if not violations:
    print(f"\n  ✓ Toutes les {len(P)} précédences sont respectées !")
else:
    for v in violations:
        print(v)

# Analyse des métriques
print(f"\n▶ Statistiques:")
print(f"  • Temps total (somme): {sum(t[j] for j in J)}")
print(f"  • Cycle Time optimal: {ct_optimal:.1f}")
print(f"  • Temps résolution: {model.Runtime:.3f}s")
print(f"  • Optimalité: {model.ObjBound:.1f} (gap: {model.MIPGap*100:.2f}%)")

# Distribution de charge
print(f"\n▶ Distribution de charge:")
total_load = 0
for k in K:
    load = sum(t[j] * x[j, k].X for j in J)
    total_load += load
    utilisation = (load / ct_optimal * 100) if ct_optimal > 0 else 0
    bar_length = int(utilisation / 10)
    bar = "█" * bar_length + "░" * (10 - bar_length)
    print(f"  Serveur {k}: {bar} {load:6.1f} / {ct_optimal:.1f}  ({utilisation:5.1f}%)")

print(f"  Charge totale: {total_load} (vérif: somme des t[j])")
print("="*70)




================ANALYSE DÉTAILLÉE - SOLUTION OPTIMALE=================

▶ Placement optimale des 20 VNF sur 4 serveurs:
  Serveur 0: VNF [0, 1, 3, 6, 7]  (Load: 722)
  Serveur 1: VNF [5, 9, 11, 13]  (Load: 718)
  Serveur 2: VNF [4, 8, 10, 14, 18, 19]  (Load: 722)
  Serveur 3: VNF [2, 12, 15, 16, 17]  (Load: 720)

▶ Vérification des précédences (validité):
  ✓ 0→5 respectée (s[0]=0.0 ≤ s[5]=1.0)  ✓ 1→6 respectée (s[1]=0.0 ≤ s[6]=0.0)  ✓ 3→7 respectée (s[3]=0.0 ≤ s[7]=0.0)  ✓ 4→8 respectée (s[4]=2.0 ≤ s[8]=2.0)  ✓ 5→9 respectée (s[5]=1.0 ≤ s[9]=1.0)  ✓ 6→10 respectée (s[6]=0.0 ≤ s[10]=2.0)  ✓ 7→11 respectée (s[7]=0.0 ≤ s[11]=1.0)  ✓ 9→12 respectée (s[9]=1.0 ≤ s[12]=3.0)  ✓ 10→12 respectée (s[10]=2.0 ≤ s[12]=3.0)  ✓ 11→13 respectée (s[11]=1.0 ≤ s[13]=1.0)  ✓ 11→14 respectée (s[11]=1.0 ≤ s[14]=2.0)  ✓ 12→15 respectée (s[12]=3.0 ≤ s[15]=3.0)  ✓ 12→16 respectée (s[12]=3.0 ≤ s[16]=3.0)  ✓ 12→17 respectée (s[12]=3.0 ≤ s[17]=3.0)  ✓ 13→19 respectée (s[13]=1.0 ≤ s[19]=2.0)  ✓ 14→18 respectée (